# 02 - Connectome-Constrained Agent Training

Train a reinforcement learning agent whose recurrent hidden state is constrained by the Central Complex connectome topology using Proximal Policy Optimization (PPO).


In [ ]:
import sys, os
from pathlib import Path

# Add project root and research/ to sys.path
cwd = Path.cwd().resolve()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / 'research' / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
    if (p / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from src.config import CONFIG
    from src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from src.environment import FlyNavigationEnv
    from src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
except ModuleNotFoundError:
    from research.src.config import CONFIG
    from research.src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from research.src.environment import FlyNavigationEnv
    from research.src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from research.src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
import torch
import matplotlib.pyplot as plt


## 1. Initialize Connectome & Network
We construct the `ConnectomeConstrainedAgent` where synaptic weights are initialized from biological synapse counts and frozen.


In [ ]:
data = ConnectomeData()
neurons_df = data.fetch_all_neurons()
conn_df = data.fetch_connectivity(min_synapses=CONFIG['synapse_threshold'])
graph = data.build_networkx_graph(conn_df)

analysis = ConnectomeAnalysis(graph, neurons_df)
cx_graph = analysis.get_subgraph_by_roi('CX')

agent = ConnectomeConstrainedAgent(
    cx_graph,
    sensory_dim=CONFIG['sensory_dim'],
    action_dim=CONFIG['action_dim'],
    config=CONFIG
)
print(f"Agent initialized with {agent.num_neurons} connectome recurrent neurons.")
print(f"Learnable recurrent weights: {agent.connectome_weight.requires_grad}")


## 2. Interactive PPO Training Loop


In [ ]:
env = FlyNavigationEnv(CONFIG)
trainer = PPOTrainer(agent, CONFIG)
logger = MetricsLogger(experiment_name='connectome_interactive')

num_episodes = 50
print(f"Training for {num_episodes} episodes...")
for ep in range(1, num_episodes + 1):
    reward, length = trainer.train_episode(env)
    logger.log_episode(ep, reward, length)
    if ep % 10 == 0 or ep == num_episodes:
        avg_r = logger.get_average_reward(last_n=min(ep, 25))
        print(f"Episode {ep:3d}/{num_episodes} | Avg Reward: {avg_r:6.2f} | Length: {length}")


## 3. Learning Curves


In [ ]:
logger.plot_learning_curve(title='Connectome-Constrained Agent Training')
